# Day 16 — MySQL Data Loading and Validation

**Project:** IPL Player Performance Analytics and Prediction  
**Goal:** Load the processed batting, bowling, and match metadata CSVs into project-owned MySQL staging tables, validate counts and candidate keys, run basic SQL checks, and automatically generate documentation from the run's actual outputs.

## Safety rules
- This notebook loads only into staging tables named `stg_ipl_batting_player_match`, `stg_ipl_bowling_player_match`, and `stg_ipl_match_metadata`.
- Each staging table is recreated on rerun. Do not use these names for unrelated data.
- It does not drop or replace final production tables.
- It does not train models or alter the source CSV files.
- Credentials are read from the project `.env` / process environment and are never printed.
- The final documentation reports actual outcomes from this run; a failed check is recorded as a failure, not silently treated as success.

Run from the project's Conda environment after Day 15's MySQL connection test works.

## 1. Find project root and prepare output folders

Expected processed inputs:
- `data/processed/batting_player_match.csv`
- `data/processed/bowling_player_match.csv`
- `data/processed/match_metadata.csv`

This cell searches the current directory and its parents for `data/processed`.

In [1]:
from pathlib import Path
import os
import json
import re
import time
import traceback
from datetime import datetime

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find data/processed from the current directory. "
        "Open Jupyter from inside the IPL project folder or update PROJECT_ROOT manually."
    )

PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports" / "day16"
DOCS_DIR = PROJECT_ROOT / "docs"
SQL_DIR = PROJECT_ROOT / "sql"

for folder in (REPORT_DIR, DOCS_DIR, SQL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RUN_STARTED_AT = datetime.now().astimezone().isoformat(timespec="seconds")
print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)
print("Day 16 report folder:", REPORT_DIR)
print("Run started:", RUN_STARTED_AT)

Project root: D:\Git\Personal\ipl-player-performance
Processed data folder: D:\Git\Personal\ipl-player-performance\data\processed
Day 16 report folder: D:\Git\Personal\ipl-player-performance\reports\day16
Run started: 2026-10-23T21:32:39+05:30


## 2. Load and inspect source files

No column names are assumed. The notebook records each file's shape, columns, missing values, and duplicate rows before loading.

In [2]:
import pandas as pd

INPUT_FILES = {
    "batting_player_match": PROCESSED_DIR / "batting_player_match.csv",
    "bowling_player_match": PROCESSED_DIR / "bowling_player_match.csv",
    "match_metadata": PROCESSED_DIR / "match_metadata.csv",
}

source_frames = {}
source_audit_rows = []
missing_inputs = []

for logical_name, path in INPUT_FILES.items():
    if not path.exists():
        missing_inputs.append(str(path))
        source_audit_rows.append({
            "dataset": logical_name, "path": str(path), "exists": False,
            "rows": None, "columns": None, "duplicate_rows": None,
            "missing_cells": None, "status": "MISSING"
        })
        continue

    frame = pd.read_csv(path, low_memory=False)
    source_frames[logical_name] = frame
    source_audit_rows.append({
        "dataset": logical_name,
        "path": str(path),
        "exists": True,
        "rows": int(len(frame)),
        "columns": int(len(frame.columns)),
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_cells": int(frame.isna().sum().sum()),
        "status": "LOADED"
    })
    print(f"\n{logical_name}: {frame.shape}")
    print("Columns:", frame.columns.tolist())
    display(frame.head(3))

source_audit = pd.DataFrame(source_audit_rows)
source_audit.to_csv(REPORT_DIR / "source_file_audit.csv", index=False)
display(source_audit)

if missing_inputs:
    raise FileNotFoundError("Required processed input file(s) missing: " + "; ".join(missing_inputs))

if any(frame.empty for frame in source_frames.values()):
    raise ValueError("At least one required input CSV is empty. Resolve this before loading.")


batting_player_match: (18768, 24)
Columns: ['match_id', 'date', 'season', 'venue', 'city', 'team_1', 'team_2', 'match_type', 'team', 'opposition', 'player_name', 'player_id', 'player_name_norm', 'team_norm', 'opposition_norm', 'runs', 'balls_faced', 'fours', 'sixes', 'batting_position', 'was_dismissed', 'dismissal_kind', 'innings_count', 'strike_rate']


,match_id,date,season,venue,city,team_1,team_2,match_type,team,opposition,...,opposition_norm,runs,balls_faced,fours,sixes,batting_position,was_dismissed,dismissal_kind,innings_count,strike_rate
0,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,NaN,Royal Challengers Bangalore,Kolkata Knight Riders,...,kolkataknightriders,9,10,1,0,8,True,run out,1,90.000000
1,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,NaN,Royal Challengers Bangalore,Kolkata Knight Riders,...,kolkataknightriders,0,2,0,0,7,True,caught,1,0.000000
2,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,NaN,Kolkata Knight Riders,Royal Challengers Bangalore,...,royalchallengersbangalore,158,73,10,13,2,False,NaN,1,216.438356



bowling_player_match: (14700, 24)
Columns: ['match_id', 'date', 'season', 'venue', 'city', 'team_1', 'team_2', 'match_type', 'team', 'opposition', 'player_name', 'player_id', 'player_name_norm', 'team_norm', 'opposition_norm', 'legal_balls', 'runs_conceded', 'wickets', 'wides', 'noballs', 'dots', 'innings_count', 'overs', 'economy']


,match_id,date,season,venue,city,team_1,team_2,match_type,team,opposition,...,opposition_norm,legal_balls,runs_conceded,wickets,wides,noballs,dots,innings_count,overs,economy
0,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,NaN,Royal Challengers Bangalore,Kolkata Knight Riders,...,kolkataknightriders,24,40,1,5,0,6,1,4.0,10.00
1,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,NaN,Kolkata Knight Riders,Royal Challengers Bangalore,...,royalchallengersbangalore,24,25,3,4,0,15,1,4.0,6.25
2,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,NaN,Kolkata Knight Riders,Royal Challengers Bangalore,...,royalchallengersbangalore,18,9,2,2,0,11,1,3.0,3.00



match_metadata: (1243, 9)
Columns: ['match_id', 'source_file', 'date', 'season', 'venue', 'city', 'team_1', 'team_2', 'match_type']


,match_id,source_file,date,season,venue,city,team_1,team_2,match_type
0,335982,335982.json,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,NaN
1,335983,335983.json,2008-04-19,2007/08,"Punjab Cricket Association Stadium, Mohali",Chandigarh,Kings XI Punjab,Chennai Super Kings,NaN
2,335984,335984.json,2008-04-19,2007/08,Feroz Shah Kotla,Delhi,Delhi Daredevils,Rajasthan Royals,NaN


,dataset,path,exists,rows,columns,duplicate_rows,missing_cells,status
0,batting_player_match,D:\Git\Personal\ipl-player-performance\data\pr...,True,18768,24,0,23666,LOADED
1,bowling_player_match,D:\Git\Personal\ipl-player-performance\data\pr...,True,14700,24,0,15295,LOADED
2,match_metadata,D:\Git\Personal\ipl-player-performance\data\pr...,True,1243,9,0,1294,LOADED


## 3. Configure database connection

Install dependencies in the active environment if needed:

```powershell
python -m pip install sqlalchemy pymysql python-dotenv
```

The loader deliberately requires a MySQL SQLAlchemy URL. It prints no username or password. If an old PostgreSQL `DATABASE_URL` is already present in the running kernel, this cell removes that kernel-only value before loading the project's `.env`.

In [3]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.engine import make_url

env_path = PROJECT_ROOT / ".env"
if env_path.exists():
    # Prevent a stale PostgreSQL URL already injected into this kernel from winning.
    os.environ.pop("DATABASE_URL", None)
    load_dotenv(env_path, override=False)
else:
    print("WARNING: No project .env file found; checking process environment.")

database_url = os.getenv("DATABASE_URL")
if not database_url:
    raise EnvironmentError(
        "DATABASE_URL is not set. Add a MySQL URL to the project .env, "
        "then rerun this cell."
    )

parsed_url = make_url(database_url)
print("Database driver:", parsed_url.drivername)
print("Host:", parsed_url.host)
print("Port:", parsed_url.port)
print("Database:", parsed_url.database)

if not parsed_url.drivername.startswith("mysql"):
    raise ValueError(
        f"Expected a MySQL SQLAlchemy URL, but found {parsed_url.drivername!r}. "
        "Update DATABASE_URL in the project .env."
    )

if parsed_url.database != "ipl_analytics":
    print("WARNING: The configured database is not 'ipl_analytics'; verify that this is intentional.")

engine = create_engine(
    database_url,
    pool_pre_ping=True,
    connect_args={"connect_timeout": 5},
)
with engine.connect() as conn:
    ping_result = conn.execute(text("SELECT 1")).scalar_one()
    server_version = conn.execute(text("SELECT VERSION()")).scalar_one()

print("SELECT 1 result:", ping_result)
print("MySQL server version:", server_version)

Database driver: mysql+pymysql
Host: localhost
Port: 3306
Database: ipl_analytics
SELECT 1 result: 1
MySQL server version: 8.0.46


## 4. Check candidate keys before loading

The key check is schema-aware. It tests `match_id + player_name` when those exact columns exist (case-insensitive matching); it also records which columns are available. It does not silently deduplicate rows.

In [4]:
def find_column(df, wanted):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    return lookup.get(wanted.lower())

candidate_key_specs = {
    "batting_player_match": ["match_id", "player_name"],
    "bowling_player_match": ["match_id", "player_name"],
    "match_metadata": ["match_id"],
}

key_rows = []
for dataset_name, frame in source_frames.items():
    requested = candidate_key_specs[dataset_name]
    resolved = [find_column(frame, name) for name in requested]
    if any(col is None for col in resolved):
        key_rows.append({
            "dataset": dataset_name,
            "candidate_key": " + ".join(requested),
            "resolved_columns": ", ".join(str(c) for c in resolved),
            "null_key_rows": None,
            "duplicate_key_rows": None,
            "status": "NOT_CHECKED_MISSING_COLUMNS"
        })
        continue

    key_df = frame[resolved]
    null_key_rows = int(key_df.isna().any(axis=1).sum())
    duplicate_key_rows = int(key_df.duplicated(keep=False).sum())
    key_rows.append({
        "dataset": dataset_name,
        "candidate_key": " + ".join(requested),
        "resolved_columns": " + ".join(str(c) for c in resolved),
        "null_key_rows": null_key_rows,
        "duplicate_key_rows": duplicate_key_rows,
        "status": "PASS" if null_key_rows == 0 and duplicate_key_rows == 0 else "REVIEW"
    })

key_audit = pd.DataFrame(key_rows)
key_audit.to_csv(REPORT_DIR / "candidate_key_checks.csv", index=False)
display(key_audit)

uncheckable = key_audit["status"].eq("NOT_CHECKED_MISSING_COLUMNS").any()
if uncheckable:
    print("WARNING: At least one candidate key could not be checked because expected columns were not found.")
if key_audit["status"].eq("REVIEW").any():
    print("WARNING: Key checks found null or duplicate key rows. They will be reported; source rows will not be removed.")

,dataset,candidate_key,resolved_columns,null_key_rows,duplicate_key_rows,status
0,batting_player_match,match_id + player_name,match_id + player_name,0,0,PASS
1,bowling_player_match,match_id + player_name,match_id + player_name,0,0,PASS
2,match_metadata,match_id,match_id,0,0,PASS


## 5. Load CSVs into MySQL staging tables

Table names are generated from a fixed allow-list, not from arbitrary source column values. Pandas/SQLAlchemy infers SQL types from the DataFrames. The staging tables are recreated on rerun; final tables are not touched.

In [5]:
STAGING_TABLES = {
    "batting_player_match": "stg_ipl_batting_player_match",
    "bowling_player_match": "stg_ipl_bowling_player_match",
    "match_metadata": "stg_ipl_match_metadata",
}

load_rows = []
load_errors = []

for dataset_name, frame in source_frames.items():
    table_name = STAGING_TABLES[dataset_name]
    try:
        # Safe for reruns only because these are dedicated staging tables.
        frame.to_sql(
            name=table_name,
            con=engine,
            if_exists="replace",
            index=False,
            chunksize=1000,
            method="multi",
        )
        load_rows.append({
            "dataset": dataset_name,
            "table_name": table_name,
            "source_rows": int(len(frame)),
            "status": "LOADED",
            "error": ""
        })
    except Exception as exc:
        load_errors.append(f"{dataset_name}: {type(exc).__name__}: {exc}")
        load_rows.append({
            "dataset": dataset_name,
            "table_name": table_name,
            "source_rows": int(len(frame)),
            "status": "FAILED",
            "error": f"{type(exc).__name__}: {exc}"
        })

load_audit = pd.DataFrame(load_rows)
load_audit.to_csv(REPORT_DIR / "staging_load_status.csv", index=False)
display(load_audit)

if load_errors:
    raise RuntimeError("One or more staging loads failed: " + " | ".join(load_errors))

,dataset,table_name,source_rows,status,error
0,batting_player_match,stg_ipl_batting_player_match,18768,LOADED,
1,bowling_player_match,stg_ipl_bowling_player_match,14700,LOADED,
2,match_metadata,stg_ipl_match_metadata,1243,LOADED,


## 6. Validate database row counts, columns, and sample queries

The row count in each staging table must equal its source CSV row count. This cell also runs read-only `COUNT(*)` checks and selects a small sample from each staging table.

In [6]:
db_validation_rows = []
sample_rows = {}

with engine.connect() as conn:
    for dataset_name, source_frame in source_frames.items():
        table_name = STAGING_TABLES[dataset_name]
        db_count = int(conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}`")).scalar_one())
        source_count = int(len(source_frame))
        columns = inspect(engine).get_columns(table_name)
        db_columns = [c["name"] for c in columns]
        source_columns = [str(c) for c in source_frame.columns]
        columns_match = source_columns == db_columns

        sample = pd.read_sql_query(text(f"SELECT * FROM `{table_name}` LIMIT 5"), conn)
        sample_rows[dataset_name] = sample

        db_validation_rows.append({
            "dataset": dataset_name,
            "table_name": table_name,
            "source_rows": source_count,
            "database_rows": db_count,
            "row_count_match": db_count == source_count,
            "source_column_count": len(source_columns),
            "database_column_count": len(db_columns),
            "column_order_match": columns_match,
            "status": "PASS" if db_count == source_count and columns_match else "REVIEW"
        })

db_validation = pd.DataFrame(db_validation_rows)
db_validation.to_csv(REPORT_DIR / "database_validation.csv", index=False)
display(db_validation)

for dataset_name, sample in sample_rows.items():
    print(f"Sample query result: {dataset_name}")
    display(sample)

if not db_validation["row_count_match"].all():
    raise AssertionError("Database row counts do not match source CSVs. See reports/day16/database_validation.csv.")

,dataset,table_name,source_rows,database_rows,row_count_match,source_column_count,database_column_count,column_order_match,status
0,batting_player_match,stg_ipl_batting_player_match,18768,18768,True,24,24,True,PASS
1,bowling_player_match,stg_ipl_bowling_player_match,14700,14700,True,24,24,True,PASS
2,match_metadata,stg_ipl_match_metadata,1243,1243,True,9,9,True,PASS


Sample query result: batting_player_match


,match_id,date,season,venue,city,team_1,team_2,match_type,team,opposition,...,opposition_norm,runs,balls_faced,fours,sixes,batting_position,was_dismissed,dismissal_kind,innings_count,strike_rate
0,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,None,Royal Challengers Bangalore,Kolkata Knight Riders,...,kolkataknightriders,9,10,1,0,8,1,run out,1,90.000000
1,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,None,Royal Challengers Bangalore,Kolkata Knight Riders,...,kolkataknightriders,0,2,0,0,7,1,caught,1,0.000000
2,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,None,Kolkata Knight Riders,Royal Challengers Bangalore,...,royalchallengersbangalore,158,73,10,13,2,0,NaN,1,216.438356
3,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,None,Royal Challengers Bangalore,Kolkata Knight Riders,...,kolkataknightriders,6,10,0,0,5,1,caught,1,60.000000
4,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,None,Kolkata Knight Riders,Royal Challengers Bangalore,...,royalchallengersbangalore,12,12,1,0,4,1,caught,1,100.000000


Sample query result: bowling_player_match


,match_id,date,season,venue,city,team_1,team_2,match_type,team,opposition,...,opposition_norm,legal_balls,runs_conceded,wickets,wides,noballs,dots,innings_count,overs,economy
0,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,None,Royal Challengers Bangalore,Kolkata Knight Riders,...,kolkataknightriders,24,40,1,5,0,6,1,4.0,10.000000
1,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,None,Kolkata Knight Riders,Royal Challengers Bangalore,...,royalchallengersbangalore,24,25,3,4,0,15,1,4.0,6.250000
2,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,None,Kolkata Knight Riders,Royal Challengers Bangalore,...,royalchallengersbangalore,18,9,2,2,0,11,1,3.0,3.000000
3,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,None,Royal Challengers Bangalore,Kolkata Knight Riders,...,kolkataknightriders,6,24,0,2,0,0,1,1.0,24.000000
4,335982,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,None,Kolkata Knight Riders,Royal Challengers Bangalore,...,royalchallengersbangalore,18,7,1,1,0,10,1,3.0,2.333333


Sample query result: match_metadata


,match_id,source_file,date,season,venue,city,team_1,team_2,match_type
0,335982,335982.json,2008-04-18,2007/08,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Kolkata Knight Riders,None
1,335983,335983.json,2008-04-19,2007/08,"Punjab Cricket Association Stadium, Mohali",Chandigarh,Kings XI Punjab,Chennai Super Kings,None
2,335984,335984.json,2008-04-19,2007/08,Feroz Shah Kotla,Delhi,Delhi Daredevils,Rajasthan Royals,None
3,335985,335985.json,2008-04-20,2007/08,Wankhede Stadium,Mumbai,Mumbai Indians,Royal Challengers Bangalore,None
4,335986,335986.json,2008-04-20,2007/08,Eden Gardens,Kolkata,Kolkata Knight Riders,Deccan Chargers,None


## 7. Write reusable SQL validation queries

The generated SQL file contains read-only checks for staging row counts. It is saved for review and is not executed as a script by this cell.

In [7]:
sql_lines = [
    "-- Day 16: read-only staging validation queries",
    "-- Generated automatically from the Day 16 notebook.",
    "SELECT COUNT(*) AS row_count FROM `stg_ipl_batting_player_match`;",
    "SELECT COUNT(*) AS row_count FROM `stg_ipl_bowling_player_match`;",
    "SELECT COUNT(*) AS row_count FROM `stg_ipl_match_metadata`;",
]

for dataset_name, frame in source_frames.items():
    key_cols = [find_column(frame, c) for c in candidate_key_specs[dataset_name]]
    if all(col is not None for col in key_cols):
        table_name = STAGING_TABLES[dataset_name]
        quoted_cols = ", ".join(f"`{col}`" for col in key_cols)
        sql_lines.extend([
            "",
            f"-- Candidate duplicate-key check for {dataset_name}",
            f"SELECT {quoted_cols}, COUNT(*) AS occurrences",
            f"FROM `{table_name}`",
            f"GROUP BY {quoted_cols}",
            "HAVING COUNT(*) > 1",
            "LIMIT 100;",
        ])

sql_path = SQL_DIR / "day16_mysql_staging_validation.sql"
sql_path.write_text("\n".join(sql_lines) + "\n", encoding="utf-8")
print("Saved SQL checks:", sql_path)

Saved SQL checks: D:\Git\Personal\ipl-player-performance\sql\day16_mysql_staging_validation.sql


## 8. Automatically generate Day 16 documentation

The Markdown report is generated from the actual source audit, key checks, load status, database validation, and connection results. It records any review items instead of claiming the entire task passed.

In [8]:
def markdown_table(df):
    if df is None or df.empty:
        return "_No rows available._"
    # Avoid requiring the optional `tabulate` package.
    cols = [str(c) for c in df.columns]
    values = df.fillna("").astype(str).values.tolist()
    widths = []
    for i, col in enumerate(cols):
        widths.append(max(len(col), *(len(row[i]) for row in values)) if values else len(col))
    header = "| " + " | ".join(cols[i].ljust(widths[i]) for i in range(len(cols))) + " |"
    separator = "| " + " | ".join("-" * widths[i] for i in range(len(cols))) + " |"
    body = [
        "| " + " | ".join(row[i].ljust(widths[i]) for i in range(len(cols))) + " |"
        for row in values
    ]
    return "\n".join([header, separator, *body])

overall_checks = {
    "mysql_connection": "PASS" if ping_result == 1 else "FAIL",
    "all_staging_loads": "PASS" if load_audit["status"].eq("LOADED").all() else "FAIL",
    "row_counts": "PASS" if db_validation["row_count_match"].all() else "FAIL",
    "column_order": "PASS" if db_validation["column_order_match"].all() else "REVIEW",
    "candidate_keys": (
        "REVIEW" if key_audit["status"].isin(["REVIEW", "NOT_CHECKED_MISSING_COLUMNS"]).any()
        else "PASS"
    ),
}
overall_status = "PASS" if all(v == "PASS" for v in overall_checks.values()) else "REVIEW REQUIRED"

doc_lines = [
    "# Day 16 — MySQL Data Loading and Validation",
    "",
    f"- **Run started:** {RUN_STARTED_AT}",
    f"- **Project root:** `{PROJECT_ROOT}`",
    f"- **Overall status:** **{overall_status}**",
    f"- **MySQL version:** `{server_version}`",
    "",
    "## Objective",
    "Load the three processed CSV datasets into dedicated MySQL staging tables and validate the load without modifying final production tables.",
    "",
    "## Check summary",
    markdown_table(pd.DataFrame([{"check": k, "status": v} for k, v in overall_checks.items()])),
    "",
    "## Source file audit",
    markdown_table(source_audit),
    "",
    "## Candidate key checks",
    markdown_table(key_audit),
    "",
    "## Staging load status",
    markdown_table(load_audit),
    "",
    "## Database validation",
    markdown_table(db_validation),
    "",
    "## Outputs generated",
    "- `reports/day16/source_file_audit.csv`",
    "- `reports/day16/candidate_key_checks.csv`",
    "- `reports/day16/staging_load_status.csv`",
    "- `reports/day16/database_validation.csv`",
    "- `sql/day16_mysql_staging_validation.sql`",
    "- `docs/day16_mysql_data_loading_validation.md`",
    "",
    "## Interpretation and limitations",
    "- Staging tables are temporary integration targets; they are not yet the final curated schema.",
    "- Candidate key results are diagnostic. Duplicate rows are not deleted automatically.",
    "- Row-count equality confirms record counts, not full semantic correctness.",
    "- Pandas infers SQL column types; a reviewed production schema and explicit constraints should be established before promoting data.",
    "- This notebook does not train models, create Django endpoints, or configure AWS.",
    "",
    "## Next steps",
    "Review candidate-key issues and schema types, then define/implement the final MySQL schema and Django database integration in the next project day.",
    "",
]
doc_path = DOCS_DIR / "day16_mysql_data_loading_validation.md"
doc_path.write_text("\n".join(doc_lines), encoding="utf-8")

task_status = pd.DataFrame([
    {"task": k, "status": v} for k, v in overall_checks.items()
])
task_status.loc[len(task_status)] = {"task": "overall", "status": overall_status}
task_status.to_csv(REPORT_DIR / "task_status.csv", index=False)

print("Generated documentation:", doc_path)
print("Generated task status:", REPORT_DIR / "task_status.csv")
print("Overall status:", overall_status)
print(doc_path.read_text(encoding="utf-8"))

Generated documentation: D:\Git\Personal\ipl-player-performance\docs\day16_mysql_data_loading_validation.md
Generated task status: D:\Git\Personal\ipl-player-performance\reports\day16\task_status.csv
Overall status: PASS
# Day 16 — MySQL Data Loading and Validation

- **Run started:** 2026-10-23T21:32:39+05:30
- **Project root:** `D:\Git\Personal\ipl-player-performance`
- **Overall status:** **PASS**
- **MySQL version:** `8.0.46`

## Objective
Load the three processed CSV datasets into dedicated MySQL staging tables and validate the load without modifying final production tables.

## Check summary
| check             | status |
| ----------------- | ------ |
| mysql_connection  | PASS   |
| all_staging_loads | PASS   |
| row_counts        | PASS   |
| column_order      | PASS   |
| candidate_keys    | PASS   |

## Source file audit
| dataset              | path                                                                           | exists | rows  | columns | duplicate_rows | missin

## 9. Day-end checklist

- [ ] MySQL connection succeeded (`SELECT 1` returned `1`).
- [ ] All three source files were found and non-empty.
- [ ] All staging loads succeeded.
- [ ] Database row counts match source row counts.
- [ ] Candidate-key warnings have been reviewed.
- [ ] `docs/day16_mysql_data_loading_validation.md` was generated.
- [ ] `reports/day16/task_status.csv` was generated.
- [ ] `.env` is listed in `.gitignore` and credentials were not committed.

**Stop here after reviewing the generated report.** Do not promote staging tables or create API endpoints until the key/schema review is complete.